# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YoyuQre/flyrank_assgn_1/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd

df=pd.read_csv('/content/content_refresh_anonymized.csv')

import pandas as pd

# Make a copy
data = df.copy()

# ----------------------------
# Binary target
# ----------------------------
y = (data["trend_direction"] == "down").astype(int)

# ----------------------------
# Drop identifiers and leakage columns
# ----------------------------
drop_cols = [
    "content_id",
    "client_id",
    "trend_direction",
    "trend_pct"
]

X = data.drop(columns=drop_cols)

# ----------------------------
# Separate numeric & categorical
# ----------------------------
num_cols = X.select_dtypes(include=["int64", "float64"]).columns
cat_cols = X.select_dtypes(include=["object"]).columns

# Fill missing values
X[num_cols] = X[num_cols].fillna(0)
X[cat_cols] = X[cat_cols].fillna("Unknown")

# One-hot encode categorical variables
X = pd.get_dummies(X, columns=cat_cols, drop_first=True)

print("Feature matrix shape:", X.shape)
print("Target shape:", y.shape)

X.head()

Feature matrix shape: (30000, 66)
Target shape: (30000,)


,search_volume,competition,cpc,word_count,char_count,impressions_90d,clicks_90d,pageviews_90d,sessions_90d,users_90d,...,char_count_tier_8000-15000,char_count_tier_<8000,char_count_tier_Unknown,impression_tier_good,impression_tier_low,impression_tier_moderate,position_tier_page_1,position_tier_page_3_5,position_tier_striking,position_tier_top_3
0,10.0,0.67,2.05,3221.0,20457.0,3803,29,22,17,16,...,False,False,False,True,False,False,False,False,True,False
1,90.0,0.01,0.05,2481.0,15562.0,15320,7,10,9,9,...,False,False,False,True,False,False,False,True,False,False
2,0.0,0.00,0.00,3515.0,23643.0,12581,11,14,11,11,...,False,False,False,True,False,False,False,True,False,False
3,10.0,0.00,0.00,0.0,0.0,11751,58,87,78,75,...,False,False,True,True,False,False,True,False,False,False
4,0.0,0.00,0.00,2803.0,17469.0,19140,24,177,145,144,...,False,False,False,True,False,False,False,True,False,False


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

The feature vector contains four main groups of features:

* **Search-performance features** (e.g., `impressions_90d`, `clicks_90d`, `ctr`, `avg_position`) describe how each content page has performed historically in search.
* **Content features** (e.g., `word_count`, `char_count`, `content_age_days`, `days_since_last_update`) describe the characteristics and freshness of the page.
* **Business context features** (e.g., `search_volume`, `competition`, `competition_level`, `main_intent`) provide information about the search environment and user intent.
* **Engagement features** (e.g., `sessions_90d`, `users_90d`, `engagement_rate`, `scroll_rate`) summarize how users interacted with the content.

Missing numerical values are filled with **0**, while missing categorical values are filled with **"Unknown"** before one-hot encoding. All selected features are historical measurements that would be available **before** making a prediction, making them suitable for a real-world content refresh recommendation system.


In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("Number of features:", X.shape[1])
print("\nSample feature names:")
print(X.columns[:20].tolist())

Number of features: 66

Sample feature names:
['search_volume', 'competition', 'cpc', 'word_count', 'char_count', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d']


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

I checked the feature set for potential data leakage before model training. Identifier columns (`content_id` and `client_id`) were removed because they do not describe page characteristics and could allow the model to memorize records. The target column (`trend_direction`) was excluded because it is the value being predicted. The `trend_pct` column was also removed because it directly measures the magnitude of the performance trend and would reveal information about the prediction target. Finally, I verified that the remaining features represent historical search, content, and engagement information that would be available before making a prediction, reducing the risk of using future or label-derived information.


In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Columns intentionally excluded to prevent leakage
excluded_cols = [
    "content_id",
    "client_id",
    "trend_direction",
    "trend_pct"
]

print("Excluded columns:")
print(excluded_cols)

print("\nLeakage check:")
for col in excluded_cols:
    print(f"{col}: {'Present' if col in df.columns else 'Missing'}")

print("\nExcluded from feature matrix:")
print(all(col not in X.columns for col in excluded_cols))

Excluded columns:
['content_id', 'client_id', 'trend_direction', 'trend_pct']

Leakage check:
content_id: Present
client_id: Present
trend_direction: Present
trend_pct: Present

Excluded from feature matrix:
True


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

| Excluded Field    | Reason for Exclusion                                                                                                                                    |
| ----------------- | ------------------------------------------------------------------------------------------------------------------------------------------------------- |
| `content_id`      | Unique identifier only. It does not describe the page and could allow the model to memorize individual records.                                         |
| `client_id`       | Client identifier rather than a page characteristic. Using it could cause the model to learn client-specific patterns instead of general relationships. |
| `trend_direction` | This is the prediction target (label). Including it as a feature would directly leak the correct answer.                                                |
| `trend_pct`       | Measures the magnitude of the observed trend and is directly related to the target, making it a source of target leakage.                               |


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.